In [1]:
from itertools import cycle
import json
import os
import google.generativeai as genai
import time
from google.api_core.exceptions import ResourceExhausted
import typing
from dotenv import load_dotenv

load_dotenv()

# List of API keys
api_keys = [
    os.getenv("GENAI_API_KEY_5"),
    os.getenv("GENAI_API_KEY_4"),
    os.getenv("GENAI_API_KEY_3"),
    os.getenv("GENAI_API_KEY_2"),
    os.getenv("GENAI_API_KEY_1"),
]
api_keys_cycle = cycle(api_keys)

# Check if all API keys are set
for i, key in enumerate(api_keys):
    if not key:
        raise ValueError(f"API key {i+1} not found. Please set the GENAI_API_KEY_{i+1} environment variable.")

class KeywordList(typing.TypedDict):
    keyword_list: list[str]

def generate_keywords(text):
    while True:
        try:
            api_key = next(api_keys_cycle)
            # print(f"Using API key: {api_key}")
            genai.configure(api_key=api_key)
            system_instruction="You are a conversational AI assistant. You are asked to generate a few keywords that summarize the content of a given text. Only give the keywords in csv format and in lowercase."
            model = genai.GenerativeModel(model_name="gemini-2.0-flash", system_instruction=system_instruction)
            generation_config = genai.GenerationConfig(
                max_output_tokens=500,
                temperature=0.1,
                candidate_count=1,
                response_mime_type="application/json",
                response_schema=KeywordList,
            )
            model_output = model.generate_content(contents=text, generation_config=generation_config)

            keywords = ""
            try:
                keywords = model_output.candidates[0].content.parts[0].text
                if not keywords.strip().startswith("{") or not keywords.strip().endswith("}"):
                    while not keywords.endswith(","):
                        keywords = keywords[:-1]
                    keywords = keywords.rstrip(",") + "]}"
                keywords = json.loads(keywords)["keyword_list"]
                keywords = ",".join([keyword.lower() for keyword in keywords])
                return keywords
            except Exception as e:
                return str(e)
        except ResourceExhausted as re:
            sleep_time = 20
            print(f"Rate limit exceeded: {re}. Waiting for {sleep_time} seconds...")
            time.sleep(sleep_time)

In [2]:
import os
from elasticsearch import Elasticsearch
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

es = Elasticsearch(os.getenv("ES_HOST"), basic_auth=(os.getenv("ES_USER"), os.getenv("ES_PASSWORD")))

# Check connection
if es.ping():
    print("Connected to Elasticsearch")
else:
    print("Failed to connect to Elasticsearch")

Connected to Elasticsearch


In [3]:
response = es.search(
    index="gumball_transcripts_with_prompts",
    query={"match_all": {}},
    size=10000
)

documents = response["hits"]["hits"]

print(f"Loaded {len(documents)} documents")

Loaded 256 documents


In [4]:
from elasticsearch.helpers import bulk
from tqdm import tqdm
import concurrent.futures

def generate_keywords_for_document(document):
    try:
        text = "\n".join([item["text"] for item in document["_source"]["transcript"]])
        keywords = generate_keywords(text)
        return {
            "_op_type": "update",
            "_index": document["_index"],
            "_id": document["_id"],
            "doc": {"keywords": keywords},
        }
    except Exception as e:
        print(f"Error processing document ID {document['_id']}: {e}")
        raise e

def process_documents(documents):
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(api_keys)) as executor:
        actions = list(tqdm(executor.map(generate_keywords_for_document, documents), total=len(documents), desc="Processing documents"))
    # Filter out None values from actions
    actions = [action for action in actions if action is not None]
    return actions

actions = process_documents(documents)
bulk(es, actions)

Processing documents:  24%|██▍       | 61/256 [00:18<00:50,  3.86it/s]

Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...


Processing documents:  25%|██▍       | 63/256 [00:19<00:57,  3.37it/s]

Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...


Processing documents:  59%|█████▉    | 152/256 [01:32<00:34,  3.05it/s]

...

Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...


Processing documents:  94%|█████████▍| 240/256 [02:37<00:03,  4.24it/s]

Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...


Processing documents: 100%|██████████| 256/256 [03:22<00:00,  1.26it/s]


(256, [])